In [1]:
import numpy as np
import matplotlib.pyplot as plt

# Desafio 1: Classificação Binária com Perceptron Treinável

In [2]:
# Matriz de treino: [Valor Normalizado (0.0 a 10.0), Frequência de Operações]
X_treino_fraude = np.array([
    [1.5, 1.0],
    [2.0, 2.0],
    [3.5, 1.5],
    [3.0, 3.0],
    [6.5, 5.0],
    [7.0, 7.0],
    [8.5, 6.0],
    [9.0, 8.0]
])

# Rótulos das classes (0: Legítima, 1: Suspeita)
y_treino_fraude = np.array([0, 0, 0, 0, 1, 1, 1, 1])

# 2. Função de Treinamento do Perceptron
def treinar_perceptron(X, y, taxa_aprendizado=0.1, epocas=50):
    n_amostras, n_features = X.shape
    # Inicialização exigida: pesos e viés com 1.0
    w = np.ones(n_features)
    b = 1.0
    
    for epoca in range(epocas):
        erros = 0
        for x_i, y_real in zip(X, y):
            y_pred = 1 if (np.dot(x_i, w) + b) >= 0 else 0
            erro = y_real - y_pred
            
            if erro != 0:
                w += taxa_aprendizado * erro * x_i
                b += taxa_aprendizado * erro
                erros += 1
                
        # Critério de parada antecipada se convergiu perfeitamente
        if erros == 0:
            print(f"Perceptron convergiu com sucesso na época {epoca + 1}!")
            break
            
    return w, b

# 3. Função de Inferência
def prever_perceptron(x_novo, w, b):
    return 1 if (np.dot(x_novo, w) + b) >= 0 else 0

# Executando o treinamento
taxa_aprendizado = 0.1
epocas_max = 100
w_calibrado, b_calibrado = treinar_perceptron(X_treino_fraude, y_treino_fraude, taxa_aprendizado, epocas_max)

print(f"\nParâmetros Calibrados do Perceptron:")
print(f"Pesos (w): w1 = {w_calibrado[0]:.4f}, w2 = {w_calibrado[1]:.4f}")
print(f"Viés (b): {b_calibrado:.4f}\n")

# 4. Amostras de Teste 
transacao_a = np.array([5.0, 4.0])
transacao_b = np.array([7.0, 6.5])

pred_a = prever_perceptron(transacao_a, w_calibrado, b_calibrado)
pred_b = prever_perceptron(transacao_b, w_calibrado, b_calibrado)

rotulo = lambda p: "1: Transação Suspeita (Risco de Fraude)" if p == 1 else "0: Transação Legítima"

print(f"Diagnóstico Transação A {transacao_a}: {rotulo(pred_a)}")
print(f"Diagnóstico Transação B {transacao_b}: {rotulo(pred_b)}")

Perceptron convergiu com sucesso na época 12!

Parâmetros Calibrados do Perceptron:
Pesos (w): w1 = -0.0000, w2 = 0.4000
Viés (b): -1.2000

Diagnóstico Transação A [5. 4.]: 1: Transação Suspeita (Risco de Fraude)
Diagnóstico Transação B [7.  6.5]: 1: Transação Suspeita (Risco de Fraude)


# Desafio 2: Predição de Risco de Churn com Classificador KNN

In [3]:
# Matriz de treino: [Dias de Inatividade, Chamados Críticos]
X_treino_churn = np.array([
    [2.0, 1.0],
    [3.0, 0.0],
    [5.0, 1.0],
    [6.0, 2.0],
    [15.0, 4.0],
    [18.0, 5.0],
    [20.0, 4.0],
    [22.0, 6.0]
])

# Rótulos (0: Baixo Risco, 1: Alto Risco)
y_treino_churn = np.array([0, 0, 0, 0, 1, 1, 1, 1])

# 2. Funções auxiliares vetorizadas de distância e KNN
def calcular_distancias(X, ponto, metrica="euclidiana"):
    if metrica == "euclidiana":
        return np.linalg.norm(X - ponto, ord=2, axis=1)
    elif metrica == "manhattan":
        return np.linalg.norm(X - ponto, ord=1, axis=1)
    else:
        raise ValueError("Métrica inválida. Escolha 'euclidiana' ou 'manhattan'.")

def knn_classificar(X, y, ponto, k=3, metrica="euclidiana"):
    distancias = calcular_distancias(X, ponto, metrica=metrica)
    vizinhos_idx = np.argsort(distancias)[:k]
    votos = y[vizinhos_idx]
    classe_predita = int(np.bincount(votos).argmax())
    return classe_predita, vizinhos_idx, distancias[vizinhos_idx]

# 3. Execução para os Clientes sob Avaliação
clientes_teste = {
    "Cliente 1": {"dados": np.array([13.0, 10.0]), "k": 3, "metrica": "euclidiana"},
    "Cliente 2": {"dados": np.array([7.0, 1.5]), "k": 5, "metrica": "euclidiana"}
}

for nome, info in clientes_teste.items():
    ponto = info["dados"]
    k = info["k"]
    metrica = info["metrica"]
    
    classe, vizinhos, dists = knn_classificar(X_treino_churn, y_treino_churn, ponto, k=k, metrica=metrica)
    rotulo_risco = "Alto Risco de Cancelamento (1)" if classe == 1 else "Baixo Risco de Cancelamento (0)"
    
    print(f"--- Avaliação: {nome} (K={k}, Métrica={metrica}) ---")
    print(f"Atributos: {ponto}")
    print(f"Classe Predita: {rotulo_risco}")
    print(f"Índices dos vizinhos mais próximos: {vizinhos}")
    print(f"Distâncias calculadas: {np.round(dists, 2)}\n")

--- Avaliação: Cliente 1 (K=3, Métrica=euclidiana) ---
Atributos: [13. 10.]
Classe Predita: Alto Risco de Cancelamento (1)
Índices dos vizinhos mais próximos: [4 5 6]
Distâncias calculadas: [6.32 7.07 9.22]

--- Avaliação: Cliente 2 (K=5, Métrica=euclidiana) ---
Atributos: [7.  1.5]
Classe Predita: Baixo Risco de Cancelamento (0)
Índices dos vizinhos mais próximos: [3 2 1 0 4]
Distâncias calculadas: [1.12 2.06 4.27 5.02 8.38]



# Desafio 3: Recomendação de Servidores Cloud por Similaridade Espacial

In [4]:
# Catálogo: [vCPUs, Memória RAM (GB), Armazenamento SSD (GB)]
catalogo_servidores = np.array([
    [2.0, 4.0, 50.0],    
    [4.0, 8.0, 100.0],   
    [8.0, 16.0, 250.0],  
    [16.0, 32.0, 500.0], 
    [32.0, 64.0, 1000.0],
    [64.0, 128.0, 2000.0]
])

nomes_servidores = [
    "Micro Instância Web",
    "Standard App Server",
    "Medium Backend & Cache",
    "Database Enterprise",
    "High Performance Computing",
    "Big Data & AI Training"
]

# 2. Função de Recomendação
def recomendar_servidores(catalogo, perfil_demandado, k=2, metrica="euclidiana"):
    distancias = calcular_distancias(catalogo, perfil_demandado, metrica=metrica)
    indices_ordenados = np.argsort(distancias)[:k]
    return indices_ordenados, distancias[indices_ordenados]

# 3. Execução para a demanda informada
perfil_demandado = np.array([12.0, 28.0, 850.0]) # vCPUs, RAM (GB), SSD (GB)
k_recomendacoes = 2

indices_rec, dists_rec = recomendar_servidores(catalogo_servidores, perfil_demandado, k=k_recomendacoes)

print(f"--- Relatório de Dimensionamento Automático (Sizing) ---")
print(f"Perfil Demandado - vCPUs: {perfil_demandado[0]}, RAM: {perfil_demandado[1]} GB, SSD: {perfil_demandado[2]} GB\n")

for pos, (idx, d) in enumerate(zip(indices_rec, dists_rec), start=1):
    servidor = catalogo_servidores[idx]
    nome_comercial = nomes_servidores[idx]
    print(f"{pos}º Lugar Recomendado:")
    print(f"  • Nome Comercial: {nome_comercial}")
    print(f"  • Especificações - vCPUs: {servidor[0]}, RAM: {servidor[1]} GB, SSD: {servidor[2]} GB")
    print(f"  • Distância Geométrica: {d:.2f}\n")

--- Relatório de Dimensionamento Automático (Sizing) ---
Perfil Demandado - vCPUs: 12.0, RAM: 28.0 GB, SSD: 850.0 GB

1º Lugar Recomendado:
  • Nome Comercial: High Performance Computing
  • Especificações - vCPUs: 32.0, RAM: 64.0 GB, SSD: 1000.0 GB
  • Distância Geométrica: 155.55

2º Lugar Recomendado:
  • Nome Comercial: Database Enterprise
  • Especificações - vCPUs: 16.0, RAM: 32.0 GB, SSD: 500.0 GB
  • Distância Geométrica: 350.05

